# Seção 11 (modelos complexos) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`02_composicao_de_modelos`](02_composicao_de_modelos.ipynb) | `Point2D.y` com `gt=0` rejeitava o próprio exemplo; texto cita `awards` sem o campo; `extra="ignore"` já é padrão |
| [`03_heranca_de_modelos`](03_heranca_de_modelos.ipynb) | `query_id: uuid4` (função como tipo): UUIDs de fora são recusados e o schema vira objeto vazio |
| [`05_projeto_solucao_instrutor`](05_projeto_solucao_instrutor.ipynb) | os bugs da seção 9 migraram para o submodelo |

## Novas

**1. Instância passada como campo não é copiada: dois "pais" compartilham o mesmo filho.**

In [1]:
from pydantic import BaseModel, ConfigDict, ValidationError


class Point(BaseModel):
    x: int = 0


class Shape(BaseModel):
    center: Point


origin = Point()
square, circle = Shape(center=origin), Shape(center=origin)
origin.x = 99
print(f"square.center.x, circle.center.x: {square.center.x}, {circle.center.x}")
print(f"mesmo objeto: {square.center is circle.center}")

square.center.x, circle.center.x: 99, 99
mesmo objeto: True


Com `revalidate_instances="never"` (padrão), a instância entra como está. Mutar o "ponto de origem" move as duas formas. Submodelos
`frozen`, ou `revalidate_instances="always"` (que cria cópias validadas).

**2. `extra="forbid"` do pai não vale para o filho.**

In [2]:
class Address(BaseModel):
    street: str


class Customer(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: str
    address: Address


print(f"typo no filho: {Customer.model_validate({'name': 'Ada', 'address': {'street': 'Rua A', 'numbr': 10}})!r}")
try:
    Customer.model_validate({"name": "Ada", "address": {"street": "Rua A"}, "phon": "1"})
except ValidationError as ex:
    print(f"typo no pai: {ex.errors()[0]['type']} em {ex.errors()[0]['loc']}")

typo no filho: Customer(name='Ada', address=Address(street='Rua A'))
typo no pai: extra_forbidden em ('phon',)


O número da casa (`numbr`) some sem erro. Configuração não se propaga entre modelos **compostos** (só por herança): use uma base
comum (`CamelBaseModel`, desta seção) para todos os submodelos.

**3. Referência circular: valida, mas não serializa.**

In [3]:
class Node(BaseModel):
    name: str
    next: "Node | None" = None


loop = Node(name="a")
loop.next = loop                                   # sem validate_assignment, ninguém impede
try:
    loop.model_dump()
except ValueError as ex:
    print(f"loop.model_dump(): ValueError: {ex}")

loop.model_dump(): ValueError: Circular reference detected (id repeated)


Estruturas com ciclo (grafo, lista duplamente ligada) funcionam em memória e quebram no primeiro dump. Para grafos, serialize
**referências** (IDs) em vez de objetos aninhados.

## Veja também

- [`pydantic/03_pydantic_avancado`](../../03_pydantic_avancado.ipynb) (caveats): subclasse num campo tipado pela base é serializada
  pelo tipo declarado; referência adiantada (`model_rebuild`).
- [`dataclass_factory/04_politicas_e_presets`](../../../dataclass_factory/04_politicas_e_presets.ipynb) (caveats): herança de `model_config` e raio de
  impacto do preset.